# SBA Loan Cleaning

This notebook keeps only records that are active, closed, or distressed per the loan condition logic defined in `cleaning_notes.ipynb`.

Business rule: retain `loancondition` values in `{'active', 'closed', 'distressed'}` and exclude non-origination and unknown statuses.

In [3]:
from pathlib import Path
import pandas as pd

input_path = Path('/home/rajiv/programming/imbalanced_feature_screener/data/sba_loans_user_cleaned.csv')
output_path = Path('/home/rajiv/programming/imbalanced_feature_screener/data/sba_loans_active_closed_distressed.csv')

df = pd.read_csv(input_path, low_memory=False)
df['loancondition'] = df['loancondition'].astype('string').str.strip().str.lower()
valid_conditions = {'active', 'closed', 'distressed'}
df_clean = df[df['loancondition'].isin(valid_conditions)].copy()
df_clean = df_clean.dropna(subset=['loancondition']).copy()
print(f'Rows before filtering: {len(df)}')
print(f'Rows after filtering: {len(df_clean)}')
print(df_clean['loancondition'].value_counts().to_string())
df_clean.to_csv(output_path, index=False)
print(f'Cleaned CSV saved to: {output_path}')


Rows before filtering: 96152
Rows after filtering: 96152
loancondition
closed        68180
active        21079
distressed     6893
Cleaned CSV saved to: /home/rajiv/programming/imbalanced_feature_screener/data/sba_loans_active_closed_distressed.csv


In [4]:
# Filter to closed/distressed and encode the target
df_imb = df_clean[df_clean['loancondition'].isin(['closed', 'distressed'])].copy()
df_imb['target'] = df_imb['loancondition'].map({'closed': 0, 'distressed': 1})
df_imb = df_imb.drop(columns=['loancondition', 'loanstatus'])

# Save to the data directory
imb_output_path = input_path.parent / 'sba_loans_imb_learn.csv'
df_imb.to_csv(imb_output_path, index=False)

print(f'Rows before target filtering: {len(df_clean)}')
print(f'Rows in imbalanced learning set: {len(df_imb)}')
print(df_imb['target'].value_counts().sort_index().to_string())
print(f'File saved to: {imb_output_path}')

Rows before target filtering: 96152
Rows in imbalanced learning set: 75073
target
0    68180
1     6893
File saved to: /home/rajiv/programming/imbalanced_feature_screener/data/sba_loans_imb_learn.csv
